In [ ]:
import nibabel as nib
import numpy as np
import os

# 设置路径
input_dir = r"C:\Users\diaoy\Desktop\test\BrainstemNavigatorv1.0\1.0\2a.BrainstemNucleiAtlas_MNI\labels_thresholded_binary_0.35"  # 存放 NIfTI 文件的目录
output_path = r"C:\Users\diaoy\Desktop\test\result\merged_brainstem_with_labels.nii.gz"  # 合并后文件的保存路径

# 加载所有脑区的 NIfTI 文件
nii_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.nii') or f.endswith('.nii.gz')]
brain_regions = {}
region_labels = {}  # 脑区编号
current_label = 1   # 编号从 1 开始

In [ ]:
for nii_file in nii_files:
    region_name = os.path.basename(nii_file).split('.')[0]  # 提取脑区名称
    brain_regions[region_name] = nib.load(nii_file)
    region_labels[region_name] = current_label  # 分配编号
    current_label += 1

# 初始化合并后的数据
shape = brain_regions[next(iter(brain_regions))].shape 
merged_data = np.zeros(shape, dtype=np.int16)

# 合并脑区数据并分配唯一编号
for region_name, nii in brain_regions.items():
    data = nii.get_fdata()
    label = region_labels[region_name]
    
    # 给当前脑区的体素分配唯一编号
    merged_data[data > 0] = label

# 删除重叠体素：对于重叠体素，保留编号较大的脑区
for x in range(merged_data.shape[0]):
    for y in range(merged_data.shape[1]):
        for z in range(merged_data.shape[2]):
            if merged_data[x, y, z] != 0:  # 如果当前体素属于某个脑区
                # 如果当前体素属于多个脑区，保留编号较大的脑区
                current_label = merged_data[x, y, z]
                for region_name, nii in brain_regions.items():
                    label = region_labels[region_name]
                    data = nii.get_fdata()
                    if data[x, y, z] > 0 and label > current_label:  # 重叠体素
                        merged_data[x, y, z] = label  # 替换为编号较大的脑区编号

# 输出每个脑区删除重叠体素后的体素数量
final_voxel_counts = {region_name: 0 for region_name in brain_regions.keys()}
for region_name, nii in brain_regions.items():
    data = nii.get_fdata()
    label = region_labels[region_name]
    # 统计删除重叠体素后的体素数量
    region_voxels = np.sum((merged_data == label) & (data > 0))
    final_voxel_counts[region_name] = region_voxels

# 删除体素数量为 0 或小于 25 的脑区
valid_regions = {region_name: count for region_name, count in final_voxel_counts.items() if count >= 25}

# 创建一个新的合并数据，仅保留有效脑区
filtered_data = np.zeros_like(merged_data, dtype=np.int16)

# 只保留有效脑区的数据
for region_name, label in region_labels.items():
    if region_name in valid_regions:
        data = brain_regions[region_name].get_fdata()
        filtered_data[data > 0] = label

# 保存合并后的文件
output_nii = nib.Nifti1Image(filtered_data, affine=brain_regions[next(iter(brain_regions))].affine)
nib.save(output_nii, output_path)

# 输出删除体素数量为 0 或小于 25 的脑区后的体素数量，并包括脑区编号
print("删除重叠体素后并删除体素数量为 0 或小于 25 的脑区后的体素数量：")
for region_name, voxel_count in valid_regions.items():
    label = region_labels[region_name]
    print(f"脑区: {region_name}, label: {label}, 体素数量: {voxel_count}")

print(f"合并并删除体素数量为 0 或小于 25 的脑区的结果已保存至 {output_path}")

In [ ]:
len(np.unique(filtered_data[filtered_data > 0]))

In [ ]:
# 文件路径
brainstem_path = r"C:\Users\diaoy\Desktop\test\result\merged_brainstem_with_labels.nii.gz"  # 脑干图谱
bna_path = r"C:\Users\diaoy\Desktop\test\BN_Atlas_246_1mm.nii.gz"  # BNA246 图谱
output_path = r"C:\Users\diaoy\Desktop\test\result\merged_brainstem_BNA246.nii.gz"  # 合并结果保存路径

In [ ]:
# 读取图谱
brainstem_img = nib.load(brainstem_path)
bna_img = nib.load(bna_path)

brainstem_data = brainstem_img.get_fdata().astype(np.int16)
bna_data = bna_img.get_fdata().astype(np.int16)

# 确定新编号范围
brainstem_labels = np.unique(brainstem_data[brainstem_data > 0])
bna_labels = np.unique(bna_data[bna_data > 0])

max_label_bna = bna_labels.max()
new_brainstem_data = brainstem_data.copy()
new_brainstem_data[brainstem_data > 0] += max_label_bna  # 为脑干图谱分配新的编号范围

# 合并数据并优先保留较大编号
merged_data = bna_data.copy()
conflict_mask = (bna_data > 0) & (new_brainstem_data > 0)  # 检测重叠体素
merged_data[conflict_mask] = np.maximum(bna_data[conflict_mask], new_brainstem_data[conflict_mask])  # 保留较大编号
merged_data[new_brainstem_data > 0] = np.maximum(merged_data[new_brainstem_data > 0], new_brainstem_data[new_brainstem_data > 0])

# 输出每个脑区的体素数量
unique_labels = np.unique(merged_data[merged_data > 0])
voxel_counts = {label: np.sum(merged_data == label) for label in unique_labels}

# 删除体素数量为 0 或小于 10 的脑区
filtered_data = np.zeros_like(merged_data, dtype=np.int16)
for label, count in voxel_counts.items():
    if count >= 0:
        filtered_data[merged_data == label] = label

# 保存最终合并后的文件
output_img = nib.Nifti1Image(filtered_data, affine=bna_img.affine)
nib.save(output_img, output_path)

# 输出结果
print("合并后的每个脑区体素数量（保留编号较大的图谱编号）：")
for label, count in voxel_counts.items():
    if count >= 0:
        print(f"脑区编号: {label}, 体素数量: {count}")

print(f"最终结果已保存至 {output_path}")

In [ ]:
from nilearn.image import resample_to_img
import nibabel as nib

# 路径
base_atlas_path = r"C:\Users\diaoy\Desktop\test\result\merged_brainstem_BNA246.nii.gz"
cerebellum_path = r"C:\Users\diaoy\Desktop\test\atl-NettekovenAsym32_space-MNI152NLin6AsymC_dseg.nii"

# 加载图像
base_img = nib.load(base_atlas_path)
cere_img = nib.load(cerebellum_path)

# 重采样小脑图谱到主图谱空间
resampled_cere_img = resample_to_img(cere_img, base_img, interpolation='nearest')

# 保存重采样结果（可选）
# nib.save(resampled_cere_img, 'resampled_cerebellum.nii.gz')


In [ ]:
import numpy as np

base_data = base_img.get_fdata().astype(np.int16)
cere_data = resampled_cere_img.get_fdata().astype(np.int16)

# 获取当前图谱最大 label
max_label = base_data.max()
print(f"当前最大 label: {max_label}")

# 小脑标签重新编号（从 247 开始）
unique_labels = np.unique(cere_data)
unique_labels = unique_labels[unique_labels > 0]  # 排除背景

label_map = {old: new for old, new in zip(unique_labels, range(max_label + 1, max_label + 1 + len(unique_labels)))}
print("小脑编号映射:", label_map)

# 插入小脑数据（避免覆盖已有脑区）
for old_label, new_label in label_map.items():
    mask = (cere_data == old_label) & (base_data == 0)
    base_data[mask] = new_label

# 保存合并结果
final_img = nib.Nifti1Image(base_data, affine=base_img.affine)
nib.save(final_img, r'C:\Users\diaoy\Desktop\test\result\final_combined_atlas_with_new_cerebellum.nii.gz')

print("✅ 新小脑图谱已成功合并！")


In [ ]:
len(np.unique(base_data))

In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd

# 加载合并后的图谱
atlas_path = r'C:\Users\diaoy\Desktop\test\result\final_combined_atlas_with_new_cerebellum.nii.gz'
atlas_img = nib.load(atlas_path)
atlas_data = atlas_img.get_fdata()
affine = atlas_img.affine

# 提取非零标签
labels = np.unique(atlas_data)
labels = labels[labels > 0]

# 存储 label 和对应的 MNI 坐标
rows = []

for label in labels:
    # 找到该 label 的所有体素索引
    coords = np.column_stack(np.where(atlas_data == label))
    if coords.size == 0:
        continue

    # 计算质心
    centroid_voxel = coords.mean(axis=0)
    centroid_mni = nib.affines.apply_affine(affine, centroid_voxel)

    rows.append({
        'Label': int(label),
        'MNI_X': round(centroid_mni[0], 2),
        'MNI_Y': round(centroid_mni[1], 2),
        'MNI_Z': round(centroid_mni[2], 2)
    })

# 转为 DataFrame
df = pd.DataFrame(rows)
# 保存为 CSV 文件
csv_output_path = r'C:\Users\diaoy\Desktop\test\result\merged_atlas_centroids.csv'
df.to_csv(csv_output_path, index=False)

print(f"每个脑区的MNI质心坐标已保存至：{csv_output_path}")
